In [5]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from datasets import load_dataset
import pandas as pd

import datasets

from transformers import AutoTokenizer, AutoModel

In [6]:
!pip install mteb

In [7]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, attention_mask, number_of_chunks):
    # hidden_state shape: [ num_chunks * num_texts, chunk_size, hidden_size]
    # attention_mask shape: [ num_chunks * num_texts, chunk_size ]
    # converting to [num_texts, num_chunks, chunk_size, hidden_size]
    re_grouped_hidden_state = []
    re_grouped_attention_mask = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped_hidden_state.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        re_grouped_attention_mask.append(
            attention_mask[text_starts_i:text_ends_i, :]
        )
        text_starts_i = text_ends_i
    return re_grouped_hidden_state, re_grouped_attention_mask

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized


def tokenize_max_tokens_strategy(tokenizer, inputs):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt")
    return tokenized

In [8]:
from mteb.similarity_functions import cos_sim
import torch
import torch.nn.functional as F

def get_hidden_states_per_batch(
        hidden_state,
        attention_mask,
        numbers_of_chunks,
        chunk_pooler="EOS"):
    """
    hidden_state - list: (batch_size, n_chunks (varying), chunk_size, hidden_size)
    attention_mask - list: (batch_size, n_chunks (varying), chunk_size)

    output should be a:
        - list of (batch_size, max_chunks, hidden_size) - tensor
        - new attention mask - tensor
    """

    def __eos_token_pool(hidden_state, attention_mask):
        # hidden_state (n_chunks, chunk_size, hidden_size)
        # attention_mask (n_chunks, chunk_size)

        # Calculate last token index for each chunk (assuming right padding where 1s are followed by 0s)
        last_token_indices = (attention_mask.sum(dim=1) - 1).long()
        batch_indices = torch.arange(hidden_state.shape[0], device=hidden_state.device)
        return hidden_state[batch_indices, last_token_indices]

    batch_size = len(hidden_state)
    chunk_size = hidden_state[-1].shape[1]
    hidden_size = hidden_state[-1].shape[-1]

    device = hidden_state[-1].device
    dtype = hidden_state[-1].dtype

    max_n_chunks = max(numbers_of_chunks) if batch_size > 0 else 0

    # In both cases (EOS or Mean Pooling), we return one vector per chunk.
    max_len = max_n_chunks

    hidden_states_padded = torch.zeros((batch_size, max_len, hidden_size), device=device, dtype=dtype)
    attn_mask_padded = torch.zeros((batch_size, max_len), device=device, dtype=attention_mask[-1].dtype)

    for t_i in range(batch_size):
        h_text = hidden_state[t_i]
        mask_text = attention_mask[t_i]
        n_chunks = h_text.shape[0]
        L = n_chunks

        if chunk_pooler == "EOS":
            h_text = __eos_token_pool(h_text, mask_text)
            hidden_states_padded[t_i, :L, :] = h_text
            attn_mask_padded[t_i, :L] = 1
        else:
            # Mean pooling of the chunk
            # h_text: (n_chunks, chunk_size, hidden_size)
            # mask_text: (n_chunks, chunk_size)

            masked_hidden_states = h_text * mask_text.unsqueeze(-1)
            sum_embeddings = masked_hidden_states.sum(dim=1)

            num_valid_tokens = mask_text.sum(dim=1)
            num_valid_tokens_clamped = torch.clamp(num_valid_tokens, min=1).unsqueeze(-1)

            chunk_embeddings = sum_embeddings / num_valid_tokens_clamped

            hidden_states_padded[t_i, :L, :] = chunk_embeddings
            attn_mask_padded[t_i, :L] = 1

    return hidden_states_padded, attn_mask_padded


class MemoryStatic:

    similarity = staticmethod(cos_sim)

    def __init__(
            self,
            batch_size,
            memory_size=5,
            hidden_size=1024,
            device="cuda",
            dtype=torch.float16):
        self.memory = torch.zeros(
            (batch_size, memory_size, hidden_size),
            device=device,
            dtype=dtype)
        self.memory_size = memory_size
        self.write_counts = torch.ones(
            (batch_size, memory_size),
            device=device,
            dtype=dtype)
        self.running_q = torch.zeros(
            (batch_size, hidden_size),
            device=device,
            dtype=dtype)

    def sample_h(self, hidden_state, attention_mask):
        batch_size, n_chunks, hidden_size = hidden_state.shape
        device = hidden_state.device
        for b in range(batch_size):
            valid_idx = attention_mask[b].nonzero(as_tuple=True)[0]  # indices of valid positions
            n_valid = valid_idx.numel()

            if n_valid == 0:
                # nothing valid: leave memory as zeros (or keep previous)
                continue

            if n_valid >= self.memory_size:
                # sample WITHOUT replacement
                perm = torch.randperm(n_valid, device=device)[:self.memory_size]
                chosen_idx = valid_idx[perm]                          # (M,)
                self.memory[b] = hidden_state[b, chosen_idx, :]
            else:
                # not enough unique vectors: take all, then repeat some to fill
                self.memory[b, :n_valid] = hidden_state[b, valid_idx, :]

                # fill the remaining slots by sampling from available indices (with replacement)
                extra = self.memory_size - n_valid
                rep = torch.randint(0, n_valid, (extra,), device=device)
                extra_idx = valid_idx[rep]
                self.memory[b, n_valid:] = hidden_state[b, extra_idx, :]

        return self.memory

    def write_memory(self, hidden_state, attention_mask):
        # hidden_state shape: [batch_size, max_chunks, hidden_size]
        # attention_mask shape: [batch_size, max_chunks]

        # first, fill the memory with random sampled hidden state vectors
        self.memory = self.sample_h(hidden_state, attention_mask)

        # next, iterate through the hidden_states and write the hidden state to the memory
        for i in range(hidden_state.shape[1]):
            h_all_batches = hidden_state[:, i, :]  # (batch_size, hidden_size)
            mask_all_batches = attention_mask[:, i]  # (batch_size)

            # only process active batches (where mask is 1)
            active_batches_idx = (mask_all_batches == 1).nonzero(as_tuple=True)[0]

            if len(active_batches_idx) == 0:
                continue # no active batches at this position, skip

            h = h_all_batches[active_batches_idx] # (num_active_batches, hidden_size)
            current_memory = self.memory[active_batches_idx] # (num_active_batches, memory_size, hidden_size)
            current_write_counts = self.write_counts[active_batches_idx] # (num_active_batches, memory_size)

            if i == 0:
                self.running_q[active_batches_idx] = h
            else:
                self.running_q[active_batches_idx] = self.running_q[active_batches_idx] * 0.65 + h * 0.35

            h_expanded = h.unsqueeze(1).expand_as(current_memory)
            sim = torch.cosine_similarity(h_expanded, current_memory, dim=2)  # (num_active_batches, memory_size)

            max_sim, max_sim_idx = torch.max(sim, dim=1)  # (num_active_batches)

            sim_prob = 1 - 1 / 2 * (1 - sim) # recalculate sim_prob for active batches

            # determine which active batches need forced rewrite (no similar memory found)
            needs_forced_rewrite = ~(sim_prob > 0.5).any(dim=1)

            # separate active batches into those that need regular update and those that need forced rewrite
            regular_update_mask = ~needs_forced_rewrite
            forced_rewrite_mask = needs_forced_rewrite

            regular_active_batches_idx = active_batches_idx[regular_update_mask]
            regular_max_sim_idx = max_sim_idx[regular_update_mask]
            regular_h = h[regular_update_mask]

            forced_active_batches_idx = active_batches_idx[forced_rewrite_mask]
            forced_current_write_counts = current_write_counts[forced_rewrite_mask]
            forced_h = h[forced_rewrite_mask]

            if len(forced_active_batches_idx) > 0:
                # rewrite the least occupied cell for these batches
                _, least_occupied_idx = torch.min(forced_current_write_counts, dim=1)
                self.memory[forced_active_batches_idx, least_occupied_idx] = forced_h
                self.write_counts[forced_active_batches_idx, least_occupied_idx] = 1 # reset count to 1 for new entry

            if len(regular_active_batches_idx) > 0:
                selected_regular = self.memory[regular_active_batches_idx, regular_max_sim_idx]
                w_counts_regular = self.write_counts[regular_active_batches_idx, regular_max_sim_idx]
                alpha_regular = 1 / w_counts_regular

                self.memory[regular_active_batches_idx, regular_max_sim_idx] = \
                    (1 - alpha_regular.unsqueeze(1)) * selected_regular + alpha_regular.unsqueeze(1) * regular_h
                self.write_counts[regular_active_batches_idx, regular_max_sim_idx] += 1

        return self.memory

    def pool_memory(self):
        eps = 1e-8
        mem = self.memory
        counts = self.write_counts

        # Do math in fp32 for stability (especially if mem is fp16)
        mem_f = mem.float()
        counts_f = counts.float().clamp_min(0.0)

        mem_f = F.normalize(mem_f, dim=-1)

        denom = counts_f.sum(dim=1, keepdim=True).clamp_min(eps)     # (B,1)
        weights = counts_f / denom                                    # (B,M)

        pooled = (weights.unsqueeze(-1) * mem_f).sum(dim=1)          # (B,D)
        pooled = F.normalize(pooled, dim=-1)

        # Cast back to original dtype if you care (optional)
        return pooled.to(mem.dtype)

In [9]:
import torch
import torch.nn.functional as F

class MemoryDynamic:
    def __init__(self, batch_size, memory_size=5, hidden_size=1024, device="cuda", dtype=torch.float16):
        self.memory_size = memory_size
        self.hidden_size = hidden_size
        self.device = device
        self.dtype = dtype

        # Fixed max storage, but variable "allocated" slots via self.n_slots
        self.memory = torch.zeros((batch_size, memory_size, hidden_size), device=device, dtype=dtype)
        self.write_counts = torch.zeros((batch_size, memory_size), device=device, dtype=torch.float32)  # keep counts fp32
        self.n_slots = torch.zeros((batch_size,), device=device, dtype=torch.long)  # how many slots are allocated

        # optional, keep if you still use it elsewhere
        self.running_q = torch.zeros((batch_size, hidden_size), device=device, dtype=dtype)

    @torch.no_grad()
    def init_one_slot(self, hidden_state, attention_mask):
        """
        Initialize with exactly ONE slot per batch item, from first valid position.
        hidden_state: (B, T, D)
        attention_mask: (B, T)
        """
        B, T, D = hidden_state.shape
        self.memory.zero_()
        self.write_counts.zero_()
        self.n_slots.zero_()

        for b in range(B):
            valid_idx = attention_mask[b].nonzero(as_tuple=True)[0]
            if valid_idx.numel() == 0:
                continue
            idx0 = valid_idx[0].item()
            self.memory[b, 0] = hidden_state[b, idx0]
            self.write_counts[b, 0] = 1.0
            self.n_slots[b] = 1

        return self.memory

    def write_memory(self, hidden_state, attention_mask, sim_threshold=0.4):
        """
        Dynamic allocation:
          - start with 1 slot
          - if no slot similar enough -> allocate new slot (until full)
        hidden_state: (B, T, D)
        attention_mask: (B, T)
        """
        B, T, D = hidden_state.shape

        # init exactly one slot
        self.init_one_slot(hidden_state, attention_mask)

        for i in range(T):
            h_all = hidden_state[:, i, :]            # (B, D)
            m_all = attention_mask[:, i]             # (B,)

            active = (m_all == 1).nonzero(as_tuple=True)[0]
            if active.numel() == 0:
                continue

            h = h_all[active].float()                # (Ba, D) do sim in fp32
            mem = self.memory[active].float()        # (Ba, M, D)
            counts = self.write_counts[active]       # (Ba, M) fp32
            n_slots = self.n_slots[active]           # (Ba,)

            # cosine sim (normalized dot)
            h_n = F.normalize(h, dim=-1)             # (Ba, D)
            mem_n = F.normalize(mem, dim=-1)         # (Ba, M, D)
            sims = torch.einsum("bd,bmd->bm", h_n, mem_n)  # (Ba, M)

            # mask out unallocated slots by setting sim to -inf
            slot_ids = torch.arange(self.memory_size, device=self.device).unsqueeze(0)  # (1, M)
            slot_mask = slot_ids < n_slots.unsqueeze(1)                                  # (Ba, M)
            sims = sims.masked_fill(~slot_mask, float("-inf"))

            max_sim, max_idx = sims.max(dim=1)       # (Ba,), (Ba,)

            # forced rewrite = "nothing is similar enough"
            forced = max_sim < sim_threshold

            # --- Allocate new slot for forced cases (if space) ---
            if forced.any():
                forced_active = active[forced]                   # batch indices in original B
                forced_h = h[forced]                             # (Bf, D)
                forced_nslots = self.n_slots[forced_active]      # (Bf,)

                has_space = forced_nslots < self.memory_size
                if has_space.any():
                    b_idx = forced_active[has_space]
                    h_new = forced_h[has_space]

                    new_slot = self.n_slots[b_idx]               # (K,)
                    # write new slot
                    self.memory[b_idx, new_slot] = h_new.to(self.dtype)
                    self.write_counts[b_idx, new_slot] = 1.0
                    self.n_slots[b_idx] = self.n_slots[b_idx] + 1

                # if no space left, overwrite least-used allocated slot
                no_space = ~has_space
                if no_space.any():
                    b_idx = forced_active[no_space]
                    h_new = forced_h[no_space]

                    counts_b = self.write_counts[b_idx]          # (K, M)
                    n_b = self.n_slots[b_idx]                    # (K,)
                    slot_ids2 = torch.arange(self.memory_size, device=self.device).unsqueeze(0)
                    mask2 = slot_ids2 < n_b.unsqueeze(1)
                    counts_masked = counts_b.masked_fill(~mask2, float("inf"))
                    overwrite_idx = counts_masked.argmin(dim=1)  # (K,)

                    self.memory[b_idx, overwrite_idx] = h_new.to(self.dtype)
                    self.write_counts[b_idx, overwrite_idx] = 1.0

            # --- Regular update for non-forced cases ---
            regular = ~forced
            if regular.any():
                reg_active = active[regular]
                reg_h = h[regular]                               # (Br, D) fp32
                reg_idx = max_idx[regular]                       # (Br,)

                # running mean update per slot: m <- m + (h - m)/count
                sel = self.memory[reg_active, reg_idx].float()   # (Br, D)
                c = self.write_counts[reg_active, reg_idx].clamp_min(1.0)  # (Br,)
                alpha = (1.0 / c).unsqueeze(1)                   # (Br,1)

                updated = sel + alpha * (reg_h - sel)
                self.memory[reg_active, reg_idx] = updated.to(self.dtype)
                self.write_counts[reg_active, reg_idx] = c + 1.0

        return self.memory

    def pool_memory(self, eps=1e-8):
        """
        Count pooling over ONLY allocated slots.
        """
        mem = F.normalize(self.memory.float(), dim=-1)            # (B,M,D)
        counts = self.write_counts.float().clamp_min(0.0)         # (B,M)

        # mask unallocated slots
        slot_ids = torch.arange(self.memory_size, device=self.device).unsqueeze(0)
        mask = slot_ids < self.n_slots.unsqueeze(1)              # (B,M)
        counts = counts * mask.float()

        denom = counts.sum(dim=1, keepdim=True).clamp_min(eps)
        w = counts / denom

        pooled = (w.unsqueeze(-1) * mem).sum(dim=1)
        pooled = F.normalize(pooled, dim=-1)
        return pooled.to(self.dtype)


In [10]:
from mteb.types import PromptType

In [12]:
from mteb import EncoderProtocol

from enum import Enum

class Strategy(Enum):
    chunking = "chunking"
    first = "first"
    max_tokens = "max_tokens"
    memory_dynamic = "memory_dynamic"
    memory_static = "memory_static"


class XMLRoBERTa(EncoderProtocol):

    name = "xlm-roberta-large"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy: Strategy, overlap, **kwargs):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        if strategy in [Strategy.memory_dynamic, Strategy.memory_static]:
            self.memory_size = kwargs.get("memory_size", 5)

        self.model.eval()

    @torch.no_grad()
    def __encode_batch(
            self,
            texts,
            **kwargs) -> np.ndarray:

        if self.strategy in [Strategy.chunking, Strategy.memory_dynamic, Strategy.memory_static]:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, texts, self.max_size, self.overlap)
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        elif self.strategy == Strategy.max_tokens:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

        if self.strategy == Strategy.memory_dynamic:
             memory = MemoryDynamic(
                len(texts),
                memory_size=self.memory_size,
                hidden_size=self.model.config.hidden_size,
                device=self.model.device,
                dtype=self.model.dtype)
        if self.strategy == Strategy.memory_static:
             memory = MemoryStatic(
                len(texts),
                memory_size=self.memory_size,
                hidden_size=self.model.config.hidden_size,
                device=self.model.device,
                dtype=self.model.dtype)

        # Batching logic
        total_len = tokenized["input_ids"].shape[0]
        max_chunk_batch = 200
        last_hidden_states_list = []

        with torch.inference_mode():
            for i in range(0, total_len, max_chunk_batch):
                batch_inputs = {
                    k: v[i : i+max_chunk_batch].to(self.model.device)
                    for k, v in tokenized.items()
                }
                outputs = self.model(**batch_inputs)
                last_hidden_states_list.append(outputs.last_hidden_state)
                del batch_inputs, outputs

            full_last_hidden_state = torch.cat(last_hidden_states_list, dim=0)
            attention_mask = tokenized["attention_mask"].to(self.model.device)

            if self.strategy in [Strategy.chunking, Strategy.memory_dynamic, Strategy.memory_static]:
                re_grouped_hidden_state, re_grouped_attention_mask = re_group_chunked_outputs(full_last_hidden_state, attention_mask, numbers_of_chunks)

            if self.strategy == Strategy.chunking:
                list_of_text_embeddings = []
                for i in range(len(re_grouped_hidden_state)):
                    hidden_states_for_text = re_grouped_hidden_state[i] # [num_chunks, chunk_size, hidden_size]
                    attention_mask_for_text = re_grouped_attention_mask[i] # [num_chunks, chunk_size]

                    masked_hidden_states = hidden_states_for_text * attention_mask_for_text.unsqueeze(-1)
                    sum_embeddings_per_chunk = masked_hidden_states.sum(dim=1)
                    num_valid_tokens_per_chunk = attention_mask_for_text.sum(dim=1)
                    num_valid_tokens_per_chunk_clamped = torch.clamp(num_valid_tokens_per_chunk, min=1)

                    # Weighted average over chunk_size dimension for each chunk
                    chunk_embeddings = sum_embeddings_per_chunk / num_valid_tokens_per_chunk_clamped.unsqueeze(-1)
                    text_embedding = chunk_embeddings.mean(dim=0)
                    list_of_text_embeddings.append(text_embedding)
                embeddings = torch.stack(list_of_text_embeddings)

            elif self.strategy in [Strategy.memory_dynamic, Strategy.memory_static]:
                hidden_states_padded, attention_padded = get_hidden_states_per_batch(
                    re_grouped_hidden_state, re_grouped_attention_mask, numbers_of_chunks, chunk_pooler=None
                )
                memory.write_memory(hidden_states_padded, attention_padded)
                embeddings = memory.pool_memory()

            elif (self.strategy == Strategy.first
                or self.strategy == Strategy.max_tokens):
                # Apply attention mask for mean pooling
                masked_output = full_last_hidden_state * attention_mask.unsqueeze(-1)
                sum_embeddings = masked_output.sum(dim=1)
                num_valid_tokens = attention_mask.sum(dim=1)
                num_valid_tokens_clamped = torch.clamp(num_valid_tokens, min=1)
                embeddings = sum_embeddings / num_valid_tokens_clamped.unsqueeze(-1)
            else:
                raise ValueError(f"Unhandled strategy for embedding calculation: {self.strategy}")

        embeddings = embeddings.detach().cpu().numpy()

        del tokenized
        if self.strategy in [Strategy.memory_dynamic, Strategy.memory_static]:
            del memory
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]
        return_hidden_state = kwargs["return_hidden_state"]

        texts = [text for batch in inputs for text in batch["text"]]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch, return_hidden_state=return_hidden_state)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings



class Qwen3_Embedding(EncoderProtocol):

    name = "Qwen3-Embedding-0.6B"
    similarity = staticmethod(cos_sim)

    def __init__(
            self,
            max_size,
            strategy,
            overlap,
            return_hidden_states=False,
            **kwargs):
        if strategy in [Strategy.memory_dynamic, Strategy.memory_static]:
            self.tokenizer = AutoTokenizer.from_pretrained(
                "Qwen/Qwen3-Embedding-0.6B",
                padding_side='right')
            self.memory_size = kwargs["memory_size"]
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(
                "Qwen/Qwen3-Embedding-0.6B",
                padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.return_hidden_states = return_hidden_states

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def preprocess_query(self, texts):
        task = 'Given a search query, retrieve relevant passages that answer the query'
        return f'Instruct: {task}\nQuery:{texts}'

    @torch.no_grad()
    def __encode_batch(self, texts, **kwargs) -> np.ndarray:
        return_hidden_state = kwargs["return_hidden_state"]
        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        elif self.strategy == Strategy.memory_dynamic:
            memory = MemoryDynamic(
                len(texts),
                memory_size=self.memory_size,
                hidden_size=1024,
                device=self.model.device,
                dtype=self.model.dtype)
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.memory_static:
            memory = MemoryStatic(
                len(texts),
                memory_size=self.memory_size,
                hidden_size=1024,
                device=self.model.device,
                dtype=self.model.dtype)
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )

        if self.strategy in [Strategy.chunking, Strategy.memory_dynamic, Strategy.memory_static]:
            numbers_of_chunks_total = sum(numbers_of_chunks)

        # Batching logic
        total_len = tokenized["input_ids"].shape[0]
        max_chunk_batch = 200

        last_hidden_states_list = []

        with torch.inference_mode():
            for i in range(0, total_len, max_chunk_batch):
                batch_inputs = {
                    k: v[i : i+max_chunk_batch].to(self.model.device)
                    for k, v in tokenized.items()
                }
                outputs = self.model(**batch_inputs)
                last_hidden_states_list.append(outputs.last_hidden_state)
                del batch_inputs, outputs # Cleanup GPU memory for next iter

            # Reconstruct full hidden state
            full_last_hidden_state = torch.cat(last_hidden_states_list, dim=0)

            # Get full attention mask on device
            attention_mask = tokenized["attention_mask"].to(self.model.device)

            if self.strategy in [Strategy.chunking, Strategy.memory_dynamic, Strategy.memory_static]:
                re_grouped_hidden_state, re_grouped_attention_mask = re_group_chunked_outputs(
                    full_last_hidden_state, attention_mask, numbers_of_chunks
                )
                if return_hidden_state:

                    del tokenized
                    torch.cuda.empty_cache()

                    return {
                        "hidden_state": [
                            h.detach().cpu().numpy()
                            for h in re_grouped_hidden_state
                        ],
                        "attention_mask": [
                            a.detach().cpu().numpy()
                            for a in re_grouped_attention_mask
                        ]
                    }

            if self.strategy == Strategy.chunking:
                embeddings = torch.stack([
                    self.__get_eos_token_embedding(t).mean(dim=0)
                    for t in re_grouped_hidden_state
                ])
            elif self.strategy in [Strategy.memory_dynamic, Strategy.memory_static]:
                hidden_states_padded, attention_padded = get_hidden_states_per_batch(
                    re_grouped_hidden_state, re_grouped_attention_mask, numbers_of_chunks
                )
                memory.write_memory(hidden_states_padded, attention_padded)
                embeddings = memory.pool_memory()
            else:
                embeddings = self.__get_eos_token_embedding(
                    full_last_hidden_state
                )

        embeddings = embeddings.detach().cpu().numpy()

        del tokenized
        if self.strategy in [Strategy.memory_dynamic, Strategy.memory_static]:
            del hidden_states_padded, attention_padded, memory
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]
        return_hidden_state = kwargs["return_hidden_state"]

        texts = [text for batch in inputs for text in batch["text"]]
        if prompt_type.value == "query":
            texts = [self.preprocess_query(t) for t in texts]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(
                batch,
                return_hidden_state=return_hidden_state)
            all_embeddings.append(batch_embeddings)
            if return_hidden_state:
                return all_embeddings

        return_embeddings = np.vstack(all_embeddings)
        print("Return Size")
        print(return_embeddings.shape)
        return return_embeddings

In [13]:
import seaborn as sns
def plot_hidden_state_similarity(model, texts):
    outputs = model.encode(
        texts,
        task_metadata=None,
        hf_split=None,
        hf_subset=None,
        prompt_type=PromptType.document,
        batch_size=1,
        return_hidden_state=True)

    embedding = model.encode(
        texts,
        task_metadata=None,
        hf_split=None,
        hf_subset=None,
        prompt_type=PromptType.document,
        batch_size=1,
        return_hidden_state=False)

    hidden_state = []
    attention_mask = []
    for chunk in outputs[0]["hidden_state"]:
        hidden_state.extend(chunk)
    hidden_state = np.vstack(hidden_state)

    for chunk in outputs[0]["attention_mask"]:
        attention_mask.extend(chunk)
    attention_mask = np.array(attention_mask)
    attention_mask = attention_mask.reshape(1, -1)[0]

    hidden_state = hidden_state[attention_mask.astype(bool)]

    emb_h_similarity = cos_sim(embedding, hidden_state)[0]

    plt.figure(figsize=(12, 6))

    # Plot token similarities using seaborn
    sns.lineplot(x=list(range(len(emb_h_similarity))), y=emb_h_similarity.numpy(), label='Similarity per token')

    # Add chunk borders
    for border in chunk_borders:
        plt.axvline(x=border, color='r', linestyle='--', label='Chunk Border' if border == chunk_borders[0] else "")

    # Calculate mean and variance for the legend
    mean_similarity = emb_h_similarity.mean().item() # .item() to get scalar from tensor
    var_similarity = emb_h_similarity.var().item() # .item() to get scalar from tensor

    # Add horizontal line for mean similarity with updated legend
    plt.axhline(y=mean_similarity, color='g', linestyle='--', label=f"Mean Similarity ($\mu$ = {mean_similarity:.2f}, $\sigma^2$ = {var_similarity:.2f})")

    plt.xlabel('Token Index')
    plt.ylabel('Cosine Similarity')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

<>:50: SyntaxWarning: invalid escape sequence '\m'
<>:50: SyntaxWarning: invalid escape sequence '\s'
<>:50: SyntaxWarning: invalid escape sequence '\m'
<>:50: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3257017231.py:50: SyntaxWarning: invalid escape sequence '\m'
  plt.axhline(y=mean_similarity, color='g', linestyle='--', label=f"Mean Similarity ($\mu$ = {mean_similarity:.2f}, $\sigma^2$ = {var_similarity:.2f})")
/tmp/ipython-input-3257017231.py:50: SyntaxWarning: invalid escape sequence '\s'
  plt.axhline(y=mean_similarity, color='g', linestyle='--', label=f"Mean Similarity ($\mu$ = {mean_similarity:.2f}, $\sigma^2$ = {var_similarity:.2f})")


# Evaluation

In [14]:
import mteb
import json
import torch
import gc
import os

tasks = ["LEMBWikimQARetrieval", "LEMBNarrativeQARetrieval"]
strategies = [Strategy.chunking, Strategy.memory_dynamic, Strategy.memory_static]
models_classes = [XMLRoBERTa, Qwen3_Embedding]

results_summary = {}
output_file = "evaluation_results_summary.json"

if os.path.exists(output_file):
    try:
        with open(output_file, "r") as f:
            results_summary = json.load(f)
    except:
        pass

for task_name in tasks:
    if task_name not in results_summary:
        results_summary[task_name] = {}

    print(f"\n{'='*20}\nLoading task: {task_name}\n{'='*20}")
    try:
        task = mteb.get_task(task_name)
    except Exception as e:
        print(f"Failed to load task {task_name}: {e}")
        continue

    for ModelClass in models_classes:
        for strategy in strategies:
            model_name = ModelClass.name
            strategy_name = strategy.value
            run_key = f"{model_name}_{strategy_name}"

            if run_key in results_summary[task_name]:
                print(f"Skipping {run_key} on {task_name} (already evaluated)")
                continue

            print(f"\nEvaluating {run_key} on {task_name}...")

            try:
                kwargs = {
                    "max_size": 512,
                    "strategy": strategy,
                    "overlap": 0,
                    "memory_size": 10
                }

                model = ModelClass(**kwargs)

                eval_result = task.evaluate(model, encode_kwargs={"batch_size": 16, "return_hidden_state": False})

                main_score = eval_result['default']['main_score']
                results_summary[task_name][run_key] = main_score
                print(f"Result for {run_key}: {main_score:.4f}")

                with open(output_file, "w") as f:
                    json.dump(results_summary, f, indent=4)

            except Exception as e:
                print(f"Error evaluating {run_key} on {task_name}: {e}")
                import traceback
                traceback.print_exc()
                results_summary[task_name][run_key] = "ERROR"

            if 'model' in locals():
                del model
            torch.cuda.empty_cache()
            gc.collect()

print("\n\nFinal Results Summary:")
print(json.dumps(results_summary, indent=4))


Loading task: LEMBWikimQARetrieval

Evaluating xlm-roberta-large_chunking on LEMBWikimQARetrieval...


Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

...

Processing queries for dataloading:   0%|          | 0/10449 [00:00<?, ? examples/s]

Return Size
(10449, 1024)


Converting corpus dict:   0%|          | 0/355 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (150191 > 131072). Running this sequence through the model will result in indexing errors


Return Size
(355, 1024)
Result for Qwen3-Embedding-0.6B_memory_static: 0.6135


Final Results Summary:
{
    "LEMBWikimQARetrieval": {
        "xlm-roberta-large_chunking": 0.12546,
        "xlm-roberta-large_memory_dynamic": 0.12606,
        "xlm-roberta-large_memory_static": 0.11651,
        "Qwen3-Embedding-0.6B_chunking": 0.76847,
        "Qwen3-Embedding-0.6B_memory_dynamic": 0.75679,
        "Qwen3-Embedding-0.6B_memory_static": 0.77871
    },
    "LEMBNarrativeQARetrieval": {
        "xlm-roberta-large_chunking": 0.02369,
        "xlm-roberta-large_memory_dynamic": 0.02373,
        "xlm-roberta-large_memory_static": 0.02367,
        "Qwen3-Embedding-0.6B_chunking": 0.61057,
        "Qwen3-Embedding-0.6B_memory_dynamic": 0.61487,
        "Qwen3-Embedding-0.6B_memory_static": 0.61352
    }
}
